# JEDI Final Boosting Experiments

Final classical-model attempt before project cleanup.

This notebook tests:

- stronger XGBoost variants
- CatBoost if installed
- dense SVD + histogram boosting
- raw vs Ridge-stacked score features
- no/low/aggressive engineered-feature dropping
- classification and ranking metrics

## Step 0: Imports And Config

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler, OneHotEncoder, StandardScaler

try:
    from xgboost import XGBClassifier
    HAS_XGBOOST = True
except ModuleNotFoundError:
    HAS_XGBOOST = False

try:
    from catboost import CatBoostClassifier
    HAS_CATBOOST = True
except ModuleNotFoundError:
    HAS_CATBOOST = False

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 180)
pd.set_option("display.max_colwidth", 140)

RANDOM_STATE = 42
N_SPLITS = 5

DATA_PATH = Path("jedi_modeling_enhanced_dataset.csv")
OUTPUT_DIR = Path("data")

print("xgboost available:", HAS_XGBOOST)
print("catboost available:", HAS_CATBOOST)

xgboost available: True
catboost available: False


## Step 1: Load Enhanced Dataset

In [2]:
df = pd.read_csv(DATA_PATH)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]:,}")
display(df.head(3))
display(df["applied"].value_counts().rename_axis("applied").reset_index(name="rows"))

Rows: 691
Columns: 90


,row_id,source_platform,source_type,scraped_at,can_apply_now,company_name,company_match_score,role_title,role_title_preference_score,role_family,role_family_preference_score,seniority,seniority_profile_fit_score,location,location_profile_fit_score,work_mode,work_mode_fit_score,employment_type,experience_min_years,experience_max_years,experience_profile_fit_score,skills_or_requirements,skills_profile_fit_score,salary_min,salary_max,salary_profile_fit_score,freshness,overall_preference_score,preference_reason,applied,apply_url,skills_or_requirements_original,skills_array,has_skills_or_requirements,skills_imputation_type,role_family_standard,seniority_standard,employment_type_standard,work_mode_standard,city_standard,country_standard,location_bucket,is_bengaluru,is_gurgaon_ncr,is_india,experience_min_bucket,salary_midpoint,salary_available,near_duplicate_key,company_name_standard,role_title_standard,experience_profile_fit_score_original,experience_profile_fit_imputed,experience_profile_fit_imputation_reason,feat_is_experience_mismatch,feat_is_senior_or_above,feat_is_lead_staff_principal,feat_is_early_career,feat_is_intern_or_new_grad,feat_is_entry_or_junior,feat_is_mid_level,feat_is_backend_or_fullstack,feat_is_ml_ai,feat_is_data_role,feat_is_devops_cloud,feat_is_qa_testing,feat_is_support_or_annotation,feat_is_security_role,feat_is_platform_infra,feat_is_llm_or_rag_role,feat_is_remote,feat_is_remote_global,feat_is_india_remote,feat_is_india_target_location,feat_is_bengaluru_or_ncr,feat_is_tier1_city,feat_is_location_mismatch,feat_is_low_salary,feat_is_mid_salary,feat_is_high_salary,feat_salary_good_but_role_bad,feat_low_salary_but_good_role,feat_good_role_good_location,feat_good_title_bad_experience,feat_remote_early_career,feat_onsite_or_hybrid_early_career,feat_bad_fit_combo,feat_strong_fit_combo,enhanced_title_skill_text,enhanced_context_text
0,1,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,Nirmata,4,"Software Engineer Intern, India",5,Backend / Cloud,4,intern,5,"Bangalore, Hybrid",4,Hybrid,5,internship,0.0,1.0,5,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,2,1200000,1200000,4,Not visible,3,"Intern role is fine, everything is good but the requirements state Kubernetes, observability tools, promethus and Go which dont match wi...",no,https://job-boards.greenhouse.io/nirmata/jobs/5040355008,Go; Java; microservices; Kubernetes; Prometheus; TimeScale; reliability; automation,"[""go"",""java"",""microservices"",""kubernetes"",""prometheus"",""timescale"",""reliability"",""automation""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,1200000,True,nirmata|software_engineer_intern_india|bengaluru,nirmata,software_engineer_intern_india,5.0,0,existing_numeric,0,0,0,1,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,1,1,0,0,1,0,0,0,1,0,0,1,0,1,"Software Engineer Intern, India | Backend / Cloud | backend | intern | Go; Java; microservices; Kubernetes; Prometheus; TimeScale; relia...","Software Engineer Intern, India | Backend / Cloud | intern | Bangalore, Hybrid | Hybrid | internship | Go; Java; microservices; Kubernet..."
1,2,Greenhouse,deterministic_scrape_candidate,27-06-2026,Yes,CloudSEK,5,SDE - Backend - Intern,5,Backend,5,intern,5,"Bengaluru, Karnataka, India",4,Hybrid,5,internship,0.0,1.0,5,Backend engineering; cybersecurity product; APIs; databases likely,4,800000,900000,3,Not visible,4,"Backend intern in Bengaluru; very relevant for fresher profile, salary is a little less and location bengaluru",yes,https://job-boards.greenhouse.io/cloudsek/jobs/5766402004,Backend engineering; cybersecurity product; APIs; databases likely,"[""backend engineering"",""cybersecurity product"",""apis"",""databases likely""]",True,original,backend,intern,internship,hybrid,bengaluru,india,bengaluru,True,False,True,0,850000,True,cloudsek|sde_backend_intern|bengaluru,cloudsek,sde_backend_intern,5.0,0,existing_numeric,0,0,0,1,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,1,1,0,0,1,0,0,0,1,0

,applied,rows
0,no,386
1,yes,305


## Step 2: Targets And Feature Contract

In [3]:
SCORE_TARGETS = [
    "role_title_preference_score",
    "role_family_preference_score",
    "seniority_profile_fit_score",
    "location_profile_fit_score",
    "work_mode_fit_score",
    "experience_profile_fit_score",
    "skills_profile_fit_score",
    "salary_profile_fit_score",
    "overall_preference_score",
]

df["applied_clean"] = df["applied"].astype(str).str.strip().str.lower()
bad = sorted(set(df["applied_clean"]) - {"yes", "no"})
if bad:
    raise ValueError(f"Invalid applied labels: {bad}")

df["target_applied"] = df["applied_clean"].map({"no": 0, "yes": 1}).astype(int)
for col in SCORE_TARGETS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

valid_rows = df["target_applied"].notna() & df[SCORE_TARGETS].notna().all(axis=1)
model_df = df.loc[valid_rows].reset_index(drop=True).copy()
y = model_df["target_applied"].astype(int)
y_scores = model_df[SCORE_TARGETS].astype(float)

print(f"Usable rows: {len(model_df):,}")
display(y.value_counts().rename_axis("target_applied").reset_index(name="rows"))

Usable rows: 691


,target_applied,rows
0,0,386
1,1,305


## Step 3: Build Feature Matrix

In [4]:
ALL_SCORE_COLUMNS = [
    c for c in model_df.columns
    if c.endswith("_score") or c in ["company_match_score", "overall_preference_score"]
]

LEAKAGE_COLUMNS = [
    "row_id",
    "applied",
    "applied_clean",
    "target_applied",
    "preference_reason",
    "apply_url",
    "source_platform",
    "source_type",
    "scraped_at",
    "near_duplicate_key",
    "company_name",
    "company_name_standard",
    "role_title_standard",
]

drop_columns = sorted(set(ALL_SCORE_COLUMNS + LEAKAGE_COLUMNS))
candidate_raw_features = [c for c in model_df.columns if c not in drop_columns]

engineered_features = [c for c in model_df.columns if c.startswith("feat_")]
base_numeric = [
    "experience_min_years",
    "salary_min",
    "salary_max",
    "salary_midpoint",
    "experience_profile_fit_imputed",
]
boolean_features = [
    "has_skills_or_requirements",
    "is_bengaluru",
    "is_gurgaon_ncr",
    "is_india",
    "salary_available",
]
categorical_features = [
    "can_apply_now",
    "freshness",
    "role_family_standard",
    "seniority_standard",
    "employment_type_standard",
    "work_mode_standard",
    "city_standard",
    "country_standard",
    "location_bucket",
    "experience_min_bucket",
    "skills_imputation_type",
    "experience_profile_fit_imputation_reason",
]

base_numeric = [c for c in base_numeric if c in model_df.columns]
boolean_features = [c for c in boolean_features if c in model_df.columns]
categorical_features = [c for c in categorical_features if c in model_df.columns]

for col in base_numeric + engineered_features:
    model_df[col] = pd.to_numeric(model_df[col], errors="coerce")

def to_binary(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    if text in {"true", "1", "yes"}:
        return 1.0
    if text in {"false", "0", "no"}:
        return 0.0
    return np.nan

for col in boolean_features:
    model_df[col] = model_df[col].map(to_binary)

for col in ["enhanced_title_skill_text", "enhanced_context_text"]:
    model_df[col] = model_df[col].fillna("").astype(str)

X_base = model_df[candidate_raw_features].copy()
X_base["enhanced_title_skill_text"] = model_df["enhanced_title_skill_text"]
X_base["enhanced_context_text"] = model_df["enhanced_context_text"]

meta = model_df[[
    "row_id",
    "company_name",
    "role_title",
    "role_family",
    "seniority",
    "location",
    "applied",
    "source_type",
]].copy()

display(pd.DataFrame({
    "group": ["base_numeric", "boolean", "categorical", "engineered"],
    "count": [len(base_numeric), len(boolean_features), len(categorical_features), len(engineered_features)],
}))

,group,count
0,base_numeric,5
1,boolean,5
2,categorical,12
3,engineered,34


## Step 4: Correlation Drop Utilities

In [5]:
def safe_corr(x, y):
    x = pd.to_numeric(pd.Series(x), errors="coerce")
    y = pd.to_numeric(pd.Series(y), errors="coerce")
    valid = x.notna() & y.notna()
    if valid.sum() < 3 or x[valid].nunique() <= 1:
        return 0.0
    return float(x[valid].corr(y[valid]))

def low_corr_features(X_train, y_train, drop_count):
    rows = []
    for col in engineered_features:
        if col in X_train.columns:
            corr = safe_corr(X_train[col], y_train)
            rows.append({"feature": col, "corr": corr, "abs_corr": abs(corr)})
    corr_df = pd.DataFrame(rows).sort_values(["abs_corr", "feature"]).reset_index(drop=True)
    return corr_df.head(min(drop_count, len(corr_df)))["feature"].tolist(), corr_df

global_drop_0, global_corr = low_corr_features(X_base, y, 0)
global_drop_5, _ = low_corr_features(X_base, y, 5)
global_drop_20, _ = low_corr_features(X_base, y, 20)

display(global_corr.sort_values("abs_corr", ascending=False).head(15))
display(global_corr.head(20))

,feature,corr,abs_corr
33,feat_is_early_career,0.634859,0.634859
32,feat_strong_fit_combo,0.607620,0.607620
31,feat_is_experience_mismatch,-0.574479,0.574479
30,feat_onsite_or_hybrid_early_career,0.515850,0.515850
29,feat_salary_good_but_role_bad,-0.420000,0.420000
28,feat_bad_fit_combo,-0.410917,0.410917
27,feat_is_entry_or_junior,0.404016,0.404016
26,feat_is_high_salary,-0.398678,0.398678
25,feat_is_senior_or_above,-0.397013,0.397013
24,feat_good_title_bad_experience,-0.396822,0.396822


,feature,corr,abs_corr
0,feat_is_support_or_annotation,-0.020366,0.020366
1,feat_is_qa_testing,0.027270,0.027270
2,feat_is_platform_infra,-0.031317,0.031317
3,feat_is_india_remote,-0.035138,0.035138
4,feat_is_data_role,0.058829,0.058829
5,feat_is_security_role,-0.073803,0.073803
6,feat_is_ml_ai,-0.090918,0.090918
7,feat_is_backend_or_fullstack,0.093927,0.093927
8,feat_is_remote_global,-0.126539,0.126539
9,feat_is_low_salary,0.146707,0.146707


## Step 5: Preprocessors

In [6]:
def sparse_preprocess(extra_numeric=None, dropped_engineered=None, text_size="standard"):
    extra_numeric = list(extra_numeric or [])
    dropped_engineered = set(dropped_engineered or [])
    active_engineered = [c for c in engineered_features if c not in dropped_engineered]
    numeric_cols = base_numeric + boolean_features + active_engineered + extra_numeric

    if text_size == "large":
        word_max, char_max = 4200, 2800
    else:
        word_max, char_max = 2600, 1800

    numeric_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", MaxAbsScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=3)),
    ])
    word_text = TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_features=word_max,
        sublinear_tf=True,
    )
    char_text = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_features=char_max,
        sublinear_tf=True,
    )
    return ColumnTransformer([
        ("numeric", numeric_pipe, numeric_cols),
        ("categorical", cat_pipe, categorical_features),
        ("word_text", word_text, "enhanced_title_skill_text"),
        ("char_text", char_text, "enhanced_context_text"),
    ], remainder="drop")

def dense_svd_preprocess(extra_numeric=None, dropped_engineered=None, n_components=160):
    return Pipeline([
        ("preprocess", sparse_preprocess(extra_numeric=extra_numeric, dropped_engineered=dropped_engineered, text_size="large")),
        ("svd", TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)),
        ("scaler", StandardScaler()),
    ])

## Step 6: Stacking Helpers

In [7]:
SCORE_BOUNDS = {"salary_profile_fit_score": (0.0, 8.0)}
PRED_SCORE_COLUMNS = ["pred_" + c for c in SCORE_TARGETS]
TRUE_SCORE_COLUMNS = ["true_" + c for c in SCORE_TARGETS]

def clip_scores(arr):
    arr = np.asarray(arr, dtype=float).copy()
    for i, target in enumerate(SCORE_TARGETS):
        low, high = SCORE_BOUNDS.get(target, (0.0, 5.0))
        arr[:, i] = np.clip(arr[:, i], low, high)
    return arr

def append_score_columns(X, score_array, prefix):
    out = X.copy()
    score_array = clip_scores(score_array)
    for i, target in enumerate(SCORE_TARGETS):
        out[f"{prefix}_{target}"] = score_array[:, i]
    return out

def make_score_model():
    return Pipeline([
        ("preprocess", sparse_preprocess(text_size="large")),
        ("ridge", Ridge(alpha=2.0)),
    ])

## Step 7: Model Factory

In [8]:
def make_model(spec, extra_numeric=None, dropped_engineered=None, scale_pos_weight=1.0):
    name = spec["name"]
    kind = spec["kind"]

    if kind == "xgb":
        if not HAS_XGBOOST:
            return None
        params = spec["params"].copy()
        params.update({
            "objective": "binary:logistic",
            "eval_metric": "logloss",
            "tree_method": "hist",
            "random_state": RANDOM_STATE,
            "n_jobs": 4,
            "scale_pos_weight": scale_pos_weight,
        })
        return Pipeline([
            ("preprocess", sparse_preprocess(extra_numeric=extra_numeric, dropped_engineered=dropped_engineered, text_size=spec.get("text_size", "standard"))),
            ("classifier", XGBClassifier(**params)),
        ])

    if kind == "cat":
        if not HAS_CATBOOST:
            return None
        return Pipeline([
            ("dense_features", dense_svd_preprocess(extra_numeric=extra_numeric, dropped_engineered=dropped_engineered, n_components=spec.get("n_components", 180))),
            ("classifier", CatBoostClassifier(
                iterations=spec.get("iterations", 650),
                depth=spec.get("depth", 4),
                learning_rate=spec.get("learning_rate", 0.035),
                l2_leaf_reg=spec.get("l2_leaf_reg", 5),
                loss_function="Logloss",
                eval_metric="Logloss",
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
                auto_class_weights="Balanced",
            )),
        ])

    if kind == "hist_svd":
        return Pipeline([
            ("dense_features", dense_svd_preprocess(extra_numeric=extra_numeric, dropped_engineered=dropped_engineered, n_components=spec.get("n_components", 160))),
            ("classifier", HistGradientBoostingClassifier(
                max_iter=spec.get("max_iter", 350),
                learning_rate=spec.get("learning_rate", 0.035),
                max_leaf_nodes=spec.get("max_leaf_nodes", 20),
                l2_regularization=spec.get("l2_regularization", 0.1),
                random_state=RANDOM_STATE,
            )),
        ])

    if kind == "logistic":
        return Pipeline([
            ("preprocess", sparse_preprocess(extra_numeric=extra_numeric, dropped_engineered=dropped_engineered, text_size=spec.get("text_size", "standard"))),
            ("classifier", LogisticRegression(
                max_iter=3000,
                solver="saga",
                penalty="elasticnet",
                l1_ratio=0.2,
                C=1.0,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=4,
            )),
        ])

    raise ValueError(f"Unknown model kind: {kind}")

## Step 8: Metrics

In [9]:
def classification_summary(model, variant, fold, y_true, proba, threshold=0.50):
    proba = np.asarray(proba, dtype=float)
    pred = (proba >= threshold).astype(int)
    return {
        "model": model,
        "variant": variant,
        "fold": fold,
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision_yes": precision_score(y_true, pred, zero_division=0),
        "recall_yes": recall_score(y_true, pred, zero_division=0),
        "f1_yes": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, proba),
        "average_precision": average_precision_score(y_true, proba),
        "log_loss": log_loss(y_true, proba, labels=[0, 1]),
        "brier_score": brier_score_loss(y_true, proba),
        "predicted_yes_count": int(pred.sum()),
    }

def ranking_summary(model, variant, y_true, proba, ks=(10, 20, 50)):
    frame = pd.DataFrame({"y": np.asarray(y_true), "p": np.asarray(proba)})
    frame = frame.sort_values("p", ascending=False).reset_index(drop=True)
    total_pos = max(int(frame["y"].sum()), 1)
    rows = []
    for k in ks:
        top = frame.head(k)
        gains = top["y"].values
        discounts = 1 / np.log2(np.arange(2, len(gains) + 2))
        dcg = float(np.sum(gains * discounts))
        ideal = np.sort(frame["y"].values)[::-1][:k]
        ideal_dcg = float(np.sum(ideal * discounts[:len(ideal)]))
        rows.append({
            "model": model,
            "variant": variant,
            "k": k,
            "precision_at_k": top["y"].mean(),
            "recall_at_k": top["y"].sum() / total_pos,
            "ndcg_at_k": dcg / ideal_dcg if ideal_dcg else 0.0,
        })
    return rows

def threshold_rows(model, variant, y_true, proba):
    rows = []
    for t in np.arange(0.20, 0.81, 0.05):
        pred = (np.asarray(proba) >= t).astype(int)
        rows.append({
            "model": model,
            "variant": variant,
            "threshold": round(float(t), 2),
            "predicted_yes_count": int(pred.sum()),
            "precision_yes": precision_score(y_true, pred, zero_division=0),
            "recall_yes": recall_score(y_true, pred, zero_division=0),
            "f1_yes": f1_score(y_true, pred, zero_division=0),
            "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        })
    return rows

## Step 9: Experiment Grid

In [10]:
MODEL_SPECS = [
    {
        "name": "xgb_conservative",
        "kind": "xgb",
        "text_size": "standard",
        "params": {
            "n_estimators": 500,
            "max_depth": 2,
            "learning_rate": 0.025,
            "min_child_weight": 5,
            "subsample": 0.90,
            "colsample_bytree": 0.90,
            "reg_alpha": 0.25,
            "reg_lambda": 5.0,
        },
    },
    {
        "name": "xgb_balanced",
        "kind": "xgb",
        "text_size": "large",
        "params": {
            "n_estimators": 650,
            "max_depth": 3,
            "learning_rate": 0.025,
            "min_child_weight": 3,
            "subsample": 0.85,
            "colsample_bytree": 0.85,
            "reg_alpha": 0.10,
            "reg_lambda": 3.0,
        },
    },
    {
        "name": "xgb_deeper",
        "kind": "xgb",
        "text_size": "large",
        "params": {
            "n_estimators": 450,
            "max_depth": 4,
            "learning_rate": 0.03,
            "min_child_weight": 2,
            "subsample": 0.80,
            "colsample_bytree": 0.80,
            "reg_alpha": 0.15,
            "reg_lambda": 4.0,
        },
    },
    {"name": "hist_svd", "kind": "hist_svd", "n_components": 180, "max_iter": 350, "learning_rate": 0.035, "max_leaf_nodes": 20, "l2_regularization": 0.10},
    {"name": "logistic_elasticnet", "kind": "logistic", "text_size": "large"},
    {"name": "catboost_svd_regularized", "kind": "cat", "n_components": 180, "iterations": 700, "depth": 4, "learning_rate": 0.035, "l2_leaf_reg": 8},
]

DROP_COUNTS = [0, 5, 20]
STACKING_MODES = ["raw", "stacked"]

display(pd.DataFrame(MODEL_SPECS)[["name", "kind"]])
print("Drop counts:", DROP_COUNTS)
print("Stacking modes:", STACKING_MODES)

,name,kind
0,xgb_conservative,xgb
1,xgb_balanced,xgb
2,xgb_deeper,xgb
3,hist_svd,hist_svd
4,logistic_elasticnet,logistic
5,catboost_svd_regularized,cat


Drop counts: [0, 5, 20]
Stacking modes: ['raw', 'stacked']


## Step 10: Run Cross-Validated Experiments

In [11]:
def run_experiments():
    outer_cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    inner_cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    detail_rows = []
    ranking_rows = []
    threshold_all = []
    pred_frames = []
    dropped_rows = []

    for fold, (train_idx, val_idx) in enumerate(outer_cv.split(X_base, y), start=1):
        X_tr = X_base.iloc[train_idx].copy()
        X_val = X_base.iloc[val_idx].copy()
        y_tr = y.iloc[train_idx].copy()
        y_val = y.iloc[val_idx].copy()
        y_tr_scores = y_scores.iloc[train_idx].copy()
        y_val_scores = y_scores.iloc[val_idx].copy()

        score_model = make_score_model()
        tr_score_oof = clip_scores(cross_val_predict(score_model, X_tr, y_tr_scores, cv=inner_cv, n_jobs=None))
        score_model.fit(X_tr, y_tr_scores)
        val_score_pred = clip_scores(score_model.predict(X_val))

        X_tr_stacked = append_score_columns(X_tr, tr_score_oof, "pred")
        X_val_stacked = append_score_columns(X_val, val_score_pred, "pred")

        fold_pred = meta.iloc[val_idx].reset_index(drop=True).copy()
        fold_pred["fold"] = fold
        fold_pred["actual_applied"] = y_val.reset_index(drop=True).values

        for drop_count in DROP_COUNTS:
            drop_features, corr_df = low_corr_features(X_tr, y_tr, drop_count)
            for feature in drop_features:
                dropped_rows.append({"fold": fold, "drop_count": drop_count, "feature": feature})

            scale_pos_weight = float((y_tr == 0).sum() / max((y_tr == 1).sum(), 1))

            for spec in MODEL_SPECS:
                for mode in STACKING_MODES:
                    if mode == "raw":
                        X_train_mode, X_val_mode = X_tr, X_val
                        extra_scores = []
                    else:
                        X_train_mode, X_val_mode = X_tr_stacked, X_val_stacked
                        extra_scores = PRED_SCORE_COLUMNS

                    estimator = make_model(
                        spec,
                        extra_numeric=extra_scores,
                        dropped_engineered=drop_features,
                        scale_pos_weight=scale_pos_weight,
                    )
                    if estimator is None:
                        continue

                    model_label = spec["name"]
                    variant = f"{mode}_drop{drop_count}"
                    estimator.fit(X_train_mode, y_tr)
                    proba = estimator.predict_proba(X_val_mode)[:, 1]

                    detail_rows.append(classification_summary(model_label, variant, fold, y_val, proba))
                    ranking_rows.extend(ranking_summary(model_label, variant, y_val, proba))
                    threshold_all.extend(threshold_rows(model_label, variant, y_val, proba))

                    fold_pred[f"{model_label}_{variant}_probability"] = proba
                    fold_pred[f"{model_label}_{variant}_prediction"] = (proba >= 0.5).astype(int)

        pred_frames.append(fold_pred)

    return (
        pd.DataFrame(detail_rows),
        pd.DataFrame(ranking_rows),
        pd.DataFrame(threshold_all),
        pd.concat(pred_frames, ignore_index=True).sort_values(["fold", "row_id"]).reset_index(drop=True),
        pd.DataFrame(dropped_rows),
    )

cv_detail, ranking_detail, threshold_detail, oof_predictions, dropped_feature_detail = run_experiments()

display(cv_detail.head())
display(ranking_detail.head())
display(threshold_detail.head())
display(oof_predictions.head())
display(dropped_feature_detail.head())

,model,variant,fold,threshold,accuracy,balanced_accuracy,precision_yes,recall_yes,f1_yes,roc_auc,average_precision,log_loss,brier_score,predicted_yes_count
0,xgb_conservative,raw_drop0,1,0.5,0.892086,0.887768,0.896552,0.852459,0.873950,0.939681,0.934805,0.303997,0.088637,58
1,xgb_conservative,stacked_drop0,1,0.5,0.899281,0.895965,0.898305,0.868852,0.883333,0.936738,0.935638,0.309625,0.091333,59
2,xgb_balanced,raw_drop0,1,0.5,0.892086,0.889554,0.883333,0.868852,0.876033,0.934426,0.937491,0.309420,0.091028,60
3,xgb_balanced,stacked_drop0,1,0.5,0.863309,0.862127,0.838710,0.852459,0.845528,0.930223,0.938907,0.331289,0.097430,62
4,xgb_deeper,raw_drop0,1,0.5,0.884892,0.883144,0.868852,0.868852,0.868852,0.935477,0.938148,0.309958,0.090928,61


,model,variant,k,precision_at_k,recall_at_k,ndcg_at_k
0,xgb_conservative,raw_drop0,10,1.00,0.163934,1.000000
1,xgb_conservative,raw_drop0,20,1.00,0.327869,1.000000
2,xgb_conservative,raw_drop0,50,0.96,0.786885,0.967734
3,xgb_conservative,stacked_drop0,10,1.00,0.163934,1.000000
4,xgb_conservative,stacked_drop0,20,1.00,0.327869,1.000000


,model,variant,threshold,predicted_yes_count,precision_yes,recall_yes,f1_yes,balanced_accuracy
0,xgb_conservative,raw_drop0,0.20,80,0.712500,0.934426,0.808511,0.819777
1,xgb_conservative,raw_drop0,0.25,74,0.770270,0.934426,0.844444,0.858239
2,xgb_conservative,raw_drop0,0.30,69,0.797101,0.901639,0.846154,0.861076
3,xgb_conservative,raw_drop0,0.35,64,0.843750,0.885246,0.864000,0.878520
4,xgb_conservative,raw_drop0,0.40,61,0.868852,0.868852,0.868852,0.883144


,row_id,company_name,role_title,role_family,seniority,location,applied,source_type,fold,actual_applied,xgb_conservative_raw_drop0_probability,xgb_conservative_raw_drop0_prediction,xgb_conservative_stacked_drop0_probability,xgb_conservative_stacked_drop0_prediction,xgb_balanced_raw_drop0_probability,xgb_balanced_raw_drop0_prediction,xgb_balanced_stacked_drop0_probability,xgb_balanced_stacked_drop0_prediction,xgb_deeper_raw_drop0_probability,xgb_deeper_raw_drop0_prediction,xgb_deeper_stacked_drop0_probability,xgb_deeper_stacked_drop0_prediction,hist_svd_raw_drop0_probability,hist_svd_raw_drop0_prediction,hist_svd_stacked_drop0_probability,hist_svd_stacked_drop0_prediction,logistic_elasticnet_raw_drop0_probability,logistic_elasticnet_raw_drop0_prediction,logistic_elasticnet_stacked_drop0_probability,logistic_elasticnet_stacked_drop0_prediction,xgb_conservative_raw_drop5_probability,xgb_conservative_raw_drop5_prediction,xgb_conservative_stacked_drop5_probability,xgb_conservative_stacked_drop5_prediction,xgb_balanced_raw_drop5_probability,xgb_balanced_raw_drop5_prediction,xgb_balanced_stacked_drop5_probability,xgb_balanced_stacked_drop5_prediction,xgb_deeper_raw_drop5_probability,xgb_deeper_raw_drop5_prediction,xgb_deeper_stacked_drop5_probability,xgb_deeper_stacked_drop5_prediction,hist_svd_raw_drop5_probability,hist_svd_raw_drop5_prediction,hist_svd_stacked_drop5_probability,hist_svd_stacked_drop5_prediction,logistic_elasticnet_raw_drop5_probability,logistic_elasticnet_raw_drop5_prediction,logistic_elasticnet_stacked_drop5_probability,logistic_elasticnet_stacked_drop5_prediction,xgb_conservative_raw_drop20_probability,xgb_conservative_raw_drop20_prediction,xgb_conservative_stacked_drop20_probability,xgb_conservative_stacked_drop20_prediction,xgb_balanced_raw_drop20_probability,xgb_balanced_raw_drop20_prediction,xgb_balanced_stacked_drop20_probability,xgb_balanced_stacked_drop20_prediction,xgb_deeper_raw_drop20_probability,xgb_deeper_raw_drop20_prediction,xgb_deeper_stacked_drop20_probability,xgb_deeper_stacked_drop20_prediction,hist_svd_raw_drop20_probability,hist_svd_raw_drop20_prediction,hist_svd_stacked_drop20_probability,hist_svd_stacked_drop20_prediction,logistic_elasticnet_raw_drop20_probability,logistic_elasticnet_raw_drop20_prediction,logistic_elasticnet_stacked_drop20_probability,logistic_elasticnet_stacked_drop20_prediction
0,14,PhonePe,"Software Engineer, Android",Frontend / Mobile,junior/mid,"Pune, India",yes,deterministic_scrape_candidate,1,1,0.641633,1,0.600766,1,0.145446,0,0.114497,0,0.176420,0,0.135286,0,0.010938,0,0.032020,0,0.851098,1,0.848934,1,0.648871,1,0.634884,1,0.162360,0,0.156192,0,0.115763,0,0.201358,0,0.160160,0,0.032041,0,0.824530,1,0.821704,1,0.619500,1,0.567072,1,0.196954,0,0.141072,0,0.216002,0,0.187460,0,0.325714,0,0.377892,0,0.885454,1,0.875923,1
1,15,Addepar,"Software Engineer, Backend – AI Platform",Backend / AI Platform,mid,"Pune, India",yes,deterministic_scrape_candidate,1,1,0.264703,0,0.324746,0,0.516584,1,0.668576,1,0.485439,0,0.622228,1,0.296949,0,0.149596,0,0.468650,0,0.473186,0,0.263777,0,0.334228,0,0.538496,1,0.622357,1,0.540078,1,0.606566,1,0.164176,0,0.120545,0,0.468856,0,0.474806,0,0.251204,0,0.315848,0,0.506440,1,0.637951,1,0.552063,1,0.535297,1,0.582090,1,0.783789,1,0.483601,0,0.489485,0
2,22,Perch Energy,Software Engineer III,Backend / Fullstack,mid,Remote India,no,deterministic_scrape_candidate,1,0,0.386568,0,0.293815,0,0.526530,1,0.528648,1,0.462784,0,0.613454,1,0.047998,0,0.024821,0,0.334020,0,0.339502,0,0.399640,0,0.342175,0,0.559204,1,0.522420,1,0.507568,1,0.540974,1,0.053442,0,0.014468,0,0.293571,0,0.302240,0,0.411083,0,0.310396,0,0.513106,1,0.503306,1,0.445145,0,0.497492,0,0.014485,0,0.053238,0,0.273167,0,0.282406,0
3,23,Truveta,Senior Software Engineer (Full-Stack),Fullstack,senior,"Hyderabad, India",no,deterministic_scrape_candidate,1,0,0.066225,0,0.141749,0,0.032426,0,0.065715,0,0.028336,0,0.057292,0,0.001131,0,0.001698,0,0.233294,0,0.238089,0,0.065142,0,0.125992,0,0.0389

,fold,drop_count,feature
0,1,5,feat_is_qa_testing
1,1,5,feat_is_platform_infra
2,1,5,feat_is_support_or_annotation
3,1,5,feat_is_india_remote
4,1,5,feat_is_data_role


## Step 11: Summaries

In [12]:
metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "precision_yes",
    "recall_yes",
    "f1_yes",
    "roc_auc",
    "average_precision",
    "log_loss",
    "brier_score",
    "predicted_yes_count",
]

summary = cv_detail.groupby(["model", "variant"])[metric_cols].agg(["mean", "std"])
summary_flat = summary.copy()
summary_flat.columns = [f"{metric}_{stat}" for metric, stat in summary_flat.columns]
summary_flat = summary_flat.reset_index().sort_values(["f1_yes_mean", "roc_auc_mean"], ascending=False)

ranking_summary = ranking_detail.groupby(["model", "variant", "k"])[["precision_at_k", "recall_at_k", "ndcg_at_k"]].mean().reset_index()
threshold_summary = (
    threshold_detail
    .groupby(["model", "variant", "threshold"])[["f1_yes", "precision_yes", "recall_yes", "balanced_accuracy", "predicted_yes_count"]]
    .mean()
    .reset_index()
)
threshold_best = (
    threshold_summary
    .sort_values(["model", "variant", "f1_yes"], ascending=[True, True, False])
    .groupby(["model", "variant"])
    .head(1)
    .sort_values("f1_yes", ascending=False)
    .reset_index(drop=True)
)

display(summary_flat.head(25))
display(threshold_best.head(25))
display(ranking_summary.sort_values(["k", "precision_at_k", "ndcg_at_k"], ascending=[True, False, False]).groupby("k").head(10))

,model,variant,accuracy_mean,accuracy_std,balanced_accuracy_mean,balanced_accuracy_std,precision_yes_mean,precision_yes_std,recall_yes_mean,recall_yes_std,f1_yes_mean,f1_yes_std,roc_auc_mean,roc_auc_std,average_precision_mean,average_precision_std,log_loss_mean,log_loss_std,brier_score_mean,brier_score_std,predicted_yes_count_mean,predicted_yes_count_std
16,xgb_balanced,stacked_drop20,0.876989,0.024577,0.875794,0.024938,0.857915,0.033720,0.865574,0.037383,0.861285,0.028242,0.932010,0.019473,0.903334,0.053056,0.349353,0.073339,0.099856,0.020626,61.6,3.209361
12,xgb_balanced,raw_drop0,0.876968,0.027698,0.874739,0.025597,0.866820,0.052191,0.855738,0.024315,0.860459,0.028176,0.933234,0.019149,0.902972,0.053213,0.341469,0.070677,0.098433,0.020390,60.4,4.159327
10,logistic_elasticnet,stacked_drop20,0.872610,0.029420,0.873568,0.028105,0.839254,0.043573,0.881967,0.021374,0.859794,0.030228,0.928040,0.024059,0.908331,0.038042,0.343759,0.072451,0.100270,0.023594,64.2,2.588436
13,xgb_balanced,raw_drop20,0.876958,0.028670,0.874398,0.027701,0.868784,0.049345,0.852459,0.034776,0.859763,0.030843,0.932636,0.018600,0.902303,0.052594,0.340395,0.068601,0.097532,0.020046,60.0,4.062019
2,hist_svd,raw_drop5,0.875550,0.033718,0.873407,0.032836,0.864257,0.055182,0.855738,0.043988,0.858804,0.036169,0.917527,0.019358,0.886493,0.039347,0.484660,0.129905,0.108998,0.027917,60.6,4.929503
25,xgb_deeper,raw_drop20,0.875519,0.028364,0.873116,0.027458,0.865825,0.048698,0.852459,0.034776,0.858331,0.030685,0.933220,0.019988,0.903515,0.051357,0.341491,0.075868,0.097991,0.022237,60.2,4.024922
15,xgb_balanced,stacked_drop0,0.874111,0.027301,0.872872,0.027517,0.854529,0.035915,0.862295,0.035916,0.858079,0.031059,0.931414,0.018564,0.901111,0.052859,0.350264,0.073656,0.100055,0.020203,61.6,2.701851
22,xgb_conservative,stacked_drop20,0.874059,0.028961,0.872465,0.028542,0.857161,0.041142,0.859016,0.031957,0.857764,0.031793,0.930765,0.018921,0.910299,0.039740,0.335815,0.057033,0.099730,0.018390,61.2,2.683282
20,xgb_conservative,raw_drop5,0.874069,0.023405,0.872125,0.023110,0.859572,0.036193,0.855738,0.031533,0.857214,0.025759,0.934153,0.020734,0.912874,0.038769,0.328532,0.057794,0.097430,0.019138,60.8,3.114482
14,xgb_balanced,raw_drop5,0.874069,0.031110,0.871460,0.029536,0.865722,0.053612,0.849180,0.029325,0.856648,0.032670,0.933446,0.020040,0.902875,0.055084,0.340235,0.070787,0.097927,0.020103,60.0,3.937004


,model,variant,threshold,predicted_yes_count,precision_yes,recall_yes,f1_yes,balanced_accuracy
0,xgb_deeper,stacked_drop0,0.35,65,0.892308,0.950820,0.920635,0.929955
1,xgb_deeper,stacked_drop5,0.40,65,0.892308,0.950820,0.920635,0.929955
2,xgb_deeper,stacked_drop20,0.35,65,0.892308,0.950820,0.920635,0.929955
3,xgb_balanced,stacked_drop20,0.35,68,0.867647,0.967213,0.914729,0.925165
4,xgb_balanced,stacked_drop0,0.40,66,0.878788,0.950820,0.913386,0.923462
5,hist_svd,stacked_drop20,0.55,62,0.903226,0.918033,0.910569,0.920055
6,xgb_balanced,raw_drop5,0.40,60,0.916667,0.901639,0.909091,0.918352
7,logistic_elasticnet,raw_drop20,0.70,58,0.931034,0.885246,0.907563,0.916649
8,logistic_elasticnet,stacked_drop20,0.70,58,0.931034,0.885246,0.907563,0.916649
9,xgb_balanced,stacked_drop5,0.35,67,0.865672,0.950820,0.906250,0.916968


,model,variant,k,precision_at_k,recall_at_k,ndcg_at_k
57,xgb_conservative,raw_drop20,10,0.980,0.160656,0.986749
54,xgb_conservative,raw_drop0,10,0.980,0.160656,0.985327
63,xgb_conservative,stacked_drop0,10,0.980,0.160656,0.977991
66,xgb_conservative,stacked_drop20,10,0.980,0.160656,0.977991
60,xgb_conservative,raw_drop5,10,0.960,0.157377,0.974025
69,xgb_conservative,stacked_drop5,10,0.960,0.157377,0.968318
6,hist_svd,raw_drop5,10,0.960,0.157377,0.950218
30,logistic_elasticnet,stacked_drop20,10,0.940,0.154098,0.959879
21,logistic_elasticnet,raw_drop20,10,0.940,0.154098,0.958826
12,hist_svd,stacked_drop20,10,0.940,0.154098,0.950263


## Step 12: Save Outputs

In [13]:
summary_path = OUTPUT_DIR / "jedi_final_boosting_experiment_summary.csv"
detail_path = OUTPUT_DIR / "jedi_final_boosting_experiment_detail.csv"
threshold_path = OUTPUT_DIR / "jedi_final_boosting_threshold_best.csv"
threshold_all_path = OUTPUT_DIR / "jedi_final_boosting_threshold_detail.csv"
ranking_path = OUTPUT_DIR / "jedi_final_boosting_ranking_summary.csv"
oof_path = OUTPUT_DIR / "jedi_final_boosting_oof_predictions.csv"
dropped_path = OUTPUT_DIR / "jedi_final_boosting_dropped_feature_detail.csv"

summary_flat.to_csv(summary_path, index=False)
cv_detail.to_csv(detail_path, index=False)
threshold_best.to_csv(threshold_path, index=False)
threshold_detail.to_csv(threshold_all_path, index=False)
ranking_summary.to_csv(ranking_path, index=False)
oof_predictions.to_csv(oof_path, index=False)
dropped_feature_detail.to_csv(dropped_path, index=False)

display(pd.DataFrame({
    "artifact": [
        "summary",
        "detail",
        "threshold_best",
        "threshold_detail",
        "ranking_summary",
        "oof_predictions",
        "dropped_feature_detail",
    ],
    "path": [
        str(summary_path),
        str(detail_path),
        str(threshold_path),
        str(threshold_all_path),
        str(ranking_path),
        str(oof_path),
        str(dropped_path),
    ],
}))

,artifact,path
0,summary,data\jedi_final_boosting_experiment_summary.csv
1,detail,data\jedi_final_boosting_experiment_detail.csv
2,threshold_best,data\jedi_final_boosting_threshold_best.csv
3,threshold_detail,data\jedi_final_boosting_threshold_detail.csv
4,ranking_summary,data\jedi_final_boosting_ranking_summary.csv
5,oof_predictions,data\jedi_final_boosting_oof_predictions.csv
6,dropped_feature_detail,data\jedi_final_boosting_dropped_feature_detail.csv


## Step 13: Final Readout

In [14]:
best = summary_flat.iloc[0]
print("Best by default-threshold F1:")
print(best[["model", "variant", "f1_yes_mean", "accuracy_mean", "roc_auc_mean", "average_precision_mean"]])

best_threshold = threshold_best.iloc[0]
print("\nBest tuned-threshold F1:")
print(best_threshold[["model", "variant", "threshold", "f1_yes", "precision_yes", "recall_yes"]])

display(
    ranking_summary
    .sort_values(["k", "precision_at_k", "ndcg_at_k"], ascending=[True, False, False])
    .groupby("k")
    .head(5)
)

Best by default-threshold F1:
model                       xgb_balanced
variant                   stacked_drop20
f1_yes_mean                     0.861285
accuracy_mean                   0.876989
roc_auc_mean                     0.93201
average_precision_mean          0.903334
Name: 16, dtype: object

Best tuned-threshold F1:
model               xgb_deeper
variant          stacked_drop0
threshold                 0.35
f1_yes                0.920635
precision_yes         0.892308
recall_yes             0.95082
Name: 0, dtype: object


,model,variant,k,precision_at_k,recall_at_k,ndcg_at_k
57,xgb_conservative,raw_drop20,10,0.980,0.160656,0.986749
54,xgb_conservative,raw_drop0,10,0.980,0.160656,0.985327
63,xgb_conservative,stacked_drop0,10,0.980,0.160656,0.977991
66,xgb_conservative,stacked_drop20,10,0.980,0.160656,0.977991
60,xgb_conservative,raw_drop5,10,0.960,0.157377,0.974025
70,xgb_conservative,stacked_drop5,20,0.950,0.311475,0.958832
64,xgb_conservative,stacked_drop0,20,0.950,0.311475,0.957384
67,xgb_conservative,stacked_drop20,20,0.950,0.311475,0.956910
13,hist_svd,stacked_drop20,20,0.950,0.311475,0.954139
4,hist_svd,raw_drop20,20,0.950,0.311475,0.949031
